#The Transformation Logic

In [0]:
catalog = "`abhi-de-ete-project-catlog`"

spark.sql(f"create schema if not exists {catalog}.gold")

In [0]:
spark.read.table(f"{catalog}.silver.crm_sales").limit(5).display()
spark.read.table(f"{catalog}.gold.dim_customers").limit(5).display()
spark.read.table(f"{catalog}.gold.dim_products").limit(5).display()


In [0]:
query = f"""
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM {catalog}.silver.crm_sales sd
LEFT JOIN {catalog}.gold.dim_products pr
    ON sd.product_number = pr.product_number
LEFT JOIN {catalog}.gold.dim_customers cu
    ON sd.customer_id = cu.customer_id;
"""
df = spark.sql(query)


In [0]:
df.limit(10).display()

#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.gold.fact_sales")

## Sanity checks of Gold table

In [0]:
spark.sql(f"""SELECT * FROM {catalog}.gold.fact_sales LIMIT 10""").limit(10).display()